In [25]:
from collections import deque


def stable_matching(men_preferences, women_preferences):
    # Lista wszystkich wolnych mężczyzn
    free_men = deque(men_preferences.keys())

    # Zapamiętuje aktualne zaręczyny
    engagements = {}

    # Zapamiętuje której kobiecie mężczyzna już się oświadczył
    proposals = {man: 0 for man in men_preferences}

    # Tworzymy ranking kobiet
    # (dzięki temu szybko sprawdzimy którego mężczyznę kobieta woli bardziej)
    women_ranking = {}

    for woman, prefs in women_preferences.items():
        ranking = {}
        for rank, man in enumerate(prefs):
            ranking[man] = rank
        women_ranking[woman] = ranking

    # Główna pętla algorytmu
    while free_men:
        # Bierzemy wolnego mężczyznę
        man = free_men.popleft()

        # Wybiera najwyżej ocenianą kobietę, której jeszcze się nie oświadczył
        woman = men_preferences[man][proposals[man]]

        # Zwiększamy licznik oświadczyn
        proposals[man] += 1

        print(f"{man} oświadcza się {woman}")

        # Jeśli kobieta jest wolna
        if woman not in engagements:
            engagements[woman] = man
            print(f"{woman} akceptuje (była wolna)\n")

        else:
            current_partner = engagements[woman]

            # Sprawdzamy którego partnera kobieta woli bardziej
            if women_ranking[woman][man] < women_ranking[woman][current_partner]:
                print(f"{woman} woli {man} zamiast {current_partner}")

                # Stary partner staje się wolny
                free_men.append(current_partner)

                # Nowe zaręczyny
                engagements[woman] = man

                print(f"{current_partner} zostaje odrzucony\n")

            else:
                print(f"{woman} odrzuca {man}\n")

                # Mężczyzna dalej jest wolny
                free_men.append(man)

    return engagements

In [26]:
# przykład

men_preferences = {
    "Adam": ["Kasia", "Ania", "Ola"],
    "Bartek": ["Ania", "Kasia", "Ola"],
    "Cezary": ["Ania", "Ola", "Kasia"]
}

women_preferences = {
    "Ania": ["Bartek", "Adam", "Cezary"],
    "Kasia": ["Adam", "Cezary", "Bartek"],
    "Ola": ["Adam", "Bartek", "Cezary"]
}

result = stable_matching(men_preferences, women_preferences)

print("STABILNE SKOJARZENIE:")
for woman, man in result.items():
    print(man, '+', woman)

Adam oświadcza się Kasia
Kasia akceptuje (była wolna)

Bartek oświadcza się Ania
Ania akceptuje (była wolna)

Cezary oświadcza się Ania
Ania odrzuca Cezary

Cezary oświadcza się Ola
Ola akceptuje (była wolna)

STABILNE SKOJARZENIE:
Adam + Kasia
Bartek + Ania
Cezary + Ola


In [27]:
# odwrotny przykład: kobiety jako grupa inicjująca

def stable_matching_women_propose(women_preferences, men_preferences):
    free_women = deque(women_preferences.keys())
    engagements = {}
    proposals = {woman: 0 for woman in women_preferences}

    men_ranking = {}

    for man, prefs in men_preferences.items():
        ranking = {}
        for rank, woman in enumerate(prefs):
            ranking[woman] = rank
        men_ranking[man] = ranking

    while free_women:
        woman = free_women.popleft()
        man = women_preferences[woman][proposals[woman]]
        proposals[woman] += 1
        
        print(f"{woman} oświadcza się {man}")

        if man not in engagements:
            engagements[man] = woman
            print(f"{man} akceptuje (był wolny)\n")
            
        else:
            current_partner = engagements[man]
            if men_ranking[man][woman] < men_ranking[man][current_partner]:
                print(f"{man} woli {woman} zamiast {current_partner}")
                engagements[man] = woman
                free_women.append(current_partner)
                print(f"{current_partner} zostaje odrzucona\n")
                
            else:
                print(f"{man} odrzuca {woman}\n")
                free_women.append(woman)

    return engagements


women_preferences = {
    "Ania": ["Bartek", "Adam", "Cezary"],
    "Kasia": ["Adam", "Bartek", "Cezary"],
    "Ola": ["Adam", "Cezary", "Bartek"]
}

men_preferences = {
    "Adam": ["Kasia", "Ania", "Ola"],
    "Bartek": ["Ania", "Kasia", "Ola"],
    "Cezary": ["Ania", "Ola", "Kasia"]
}

result1 = stable_matching_women_propose(women_preferences, men_preferences)

print("STABILNE SKOJARZENIE:")
for man, woman in result1.items():
    print(woman, '+', man)

Ania oświadcza się Bartek
Bartek akceptuje (był wolny)

Kasia oświadcza się Adam
Adam akceptuje (był wolny)

Ola oświadcza się Adam
Adam odrzuca Ola

Ola oświadcza się Cezary
Cezary akceptuje (był wolny)

STABILNE SKOJARZENIE:
Ania + Bartek
Kasia + Adam
Ola + Cezary


In [28]:
# przykład, w którym w zależnosci od tego która grupa inicjuje, otrzymamy inny rezultat

men_preferences1 = {
    "Marek": ["Ela", "Gosia", "Hania"],
    "Piotr": ["Gosia", "Ela", "Hania"],
    "Jan": ["Ela", "Gosia", "Hania"]
}

women_preferences1 = {
    "Ela": ["Piotr", "Marek", "Jan"],
    "Gosia": ["Marek", "Piotr", "Jan"],
    "Hania": ["Marek", "Piotr", "Jan"]
}

In [29]:
result_m = stable_matching(men_preferences1, women_preferences1)
print("STABILNE SKOJARZENIE (inicjują mężczyźni):")
for woman, man in result_m.items():
    print(man, '+', woman)

Marek oświadcza się Ela
Ela akceptuje (była wolna)

Piotr oświadcza się Gosia
Gosia akceptuje (była wolna)

Jan oświadcza się Ela
Ela odrzuca Jan

Jan oświadcza się Gosia
Gosia odrzuca Jan

Jan oświadcza się Hania
Hania akceptuje (była wolna)

STABILNE SKOJARZENIE (inicjują mężczyźni):
Marek + Ela
Piotr + Gosia
Jan + Hania


In [30]:
result_w = stable_matching_women_propose(women_preferences1, men_preferences1)
print("STABILNE SKOJARZENIE (inicjują kobiety):")
for man, woman in result_w.items():
    print(woman, '+', man)

Ela oświadcza się Piotr
Piotr akceptuje (był wolny)

Gosia oświadcza się Marek
Marek akceptuje (był wolny)

Hania oświadcza się Marek
Marek odrzuca Hania

Hania oświadcza się Piotr
Piotr odrzuca Hania

Hania oświadcza się Jan
Jan akceptuje (był wolny)

STABILNE SKOJARZENIE (inicjują kobiety):
Ela + Piotr
Gosia + Marek
Hania + Jan
